In [1]:
import pandas as pd

pd.set_option("display.float_format", "{:.3f}".format)

df = pd.read_csv("en.openfoodfacts.org.products.csv", sep="\t", on_bad_lines="skip")

df.head()

C:\Users\De\AppData\Local\Temp\ipykernel_16644\3599398486.py:5: DtypeWarning: Columns (0,11,17,32,33,34,35,36,46,51,52,53,54,57,68,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("en.openfoodfacts.org.products.csv", sep="\t", on_bad_lines="skip")


,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,...,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g,water_100g
0,54,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1582569031,2020-02-24T18:30:31Z,1733085204,2024-12-01T20:33:24Z,NaN,1740205422.000,2025-02-22T06:23:42Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,63,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1673620307,2023-01-13T14:31:47Z,1750061386,2025-06-16T08:09:46Z,bodysupport,1750061386.000,2025-06-16T08:09:46Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,114,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1580066482,2020-01-26T19:21:22Z,1751035658,2025-06-27T14:47:38Z,teolemon,1751035658.000,2025-06-27T14:47:38Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,431,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1714301712,2024-04-28T10:55:12Z,1714301721,2024-04-28T10:55:21Z,kiliweb,1714301721.000,2024-04-28T10:55:21Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,105,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1572117743,2019-10-26T19:22:23Z,1738073570,2025-01-28T14:12:50Z,NaN,1743653496.000,2025-04-03T04:11:36Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4501355 entries, 0 to 4501354
Data columns (total 210 columns):
 #    Column                            Non-Null Count    Dtype  
---   ------                            --------------    -----  
 0    code                              4501355 non-null  object 
 1    url                               4501355 non-null  object 
 2    creator                           4501349 non-null  object 
 3    created_t                         4501355 non-null  int64  
 4    created_datetime                  4501355 non-null  object 
 5    last_modified_t                   4501355 non-null  int64  
 6    last_modified_datetime            4501355 non-null  object 
 7    last_modified_by                  4362794 non-null  object 
 8    last_updated_t                    4501160 non-null  float64
 9    last_updated_datetime             4501160 non-null  object 
 10   product_name                      4166210 non-null  object 
 11   abbreviated_product_na

In [3]:
import numpy as np

INVALID = {"?", ".", ",", "n-a", "na", "none", "null", "0", "en:null", "en:none"}

def clean_series(s):
    s = s.str.strip()
    s = s.replace("", np.nan)
    s = s.where(~s.str.lower().isin(INVALID), np.nan)
    s = s.where(~s.str.fullmatch(r"[?,.\-/ ]+", na=False), np.nan)
    return s



df["brands"] = df["brands"].fillna(df["brands_en"].fillna(df["brands_tags"]))

df = df.drop(columns=["brands_tags", "brands_en"])


df["labels_en"] = (df["labels_en"].fillna(df["labels"].fillna(df["labels_tags"])))

df = df.drop(columns=["labels", "labels_tags"])


df["countries_en"] = (df["countries_en"].fillna(df["countries_tags"].fillna(df["countries"])))

df = df.drop(columns=["countries", "countries_tags"])


df["food_groups_en"] = (df["food_groups_en"].fillna(df["food_groups_tags"].fillna(df["food_groups"])))

df = df.drop(columns=["food_groups", "food_groups_tags"])


df["main_category_en"] = (df["main_category_en"].fillna(df["main_category"]))

df = df.drop(columns=["main_category"])


df = (df.rename(columns={"labels_en" : "labels"
                        , "countries_en" : "countries"
                        , "food_groups_en" : "food_groups"
                        , "main_category_en" : "main_category"}))


df['additives_tags'] = (df['additives_tags'].fillna(df['additives_en']))

df = df.drop(columns=["additives_en"])


df['ingredients_tags'] = (df['ingredients_tags'].fillna(df['ingredients_text']))

df = df.drop(columns=["ingredients_text"])


df['manufacturing_places_tags'] = (df['manufacturing_places_tags'].fillna(df['manufacturing_places']))

df = df.drop(columns=["manufacturing_places"])


df['packaging_en'] = (df['packaging_en'].fillna(df['packaging']).fillna(df['packaging_tags']))

df = df.drop(columns=["packaging", "packaging_tags"])



df = df.drop(columns=["states", "states_tags"])
df = df.rename(columns={"states_en": "states"})

for c in ["categories", "categories_tags", "categories_en"]:
    df[c] = clean_series(df[c].astype(str).where(df[c].notna(), np.nan))

df["categories_en"] = (df["categories_en"].fillna(df["categories_tags"]).fillna(df["categories"]))

df = df.drop(columns=["categories_tags", "categories"])


df = df.drop(columns=["emb_codes_tags"])

df = df.drop(columns=["allergens_en"])





df["product_name"] = df["product_name"].fillna(df["abbreviated_product_name"]).fillna(df["generic_name"])

drop_col = ["abbreviated_product_name", "generic_name"]

df = df.drop(columns=drop_col)


drop_col = ["origins", "origins_tags"]

df = df.drop(columns=drop_col)


df = df.drop(columns="cities")


df["traces"] = df["traces"].fillna(df["traces_tags"]).fillna(df["traces_en"])

drop_col = ["traces_tags", "traces_en"]

df = df.drop(columns=drop_col)


drop_col = ["quantity", "serving_size"]

df = df.drop(columns=drop_col)

df = df.drop(columns=["energy-kj_100g"])

df = df.drop_duplicates(subset=["code"], keep=False)             

In [4]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 4501223 entries, 0 to 4501354
Data columns (total 180 columns):
 #    Column                            Non-Null Count    Dtype  
---   ------                            --------------    -----  
 0    code                              4501223 non-null  object 
 1    url                               4501223 non-null  object 
 2    creator                           4501217 non-null  object 
 3    created_t                         4501223 non-null  int64  
 4    created_datetime                  4501223 non-null  object 
 5    last_modified_t                   4501223 non-null  int64  
 6    last_modified_datetime            4501223 non-null  object 
 7    last_modified_by                  4362662 non-null  object 
 8    last_updated_t                    4501071 non-null  float64
 9    last_updated_datetime             4501071 non-null  object 
 10   product_name                      4169000 non-null  object 
 11   packaging_en               

In [5]:
import pandas as pd

# 1. Wir definieren unsere Simulations-Funktion
def teste_fuellgrenze(df, prozent_grenze):
    
    # Absolute Grenze in Zeilen berechnen
    mindest_zeilen = int(len(df) * prozent_grenze)
    
    # Zählt die echten (nicht-leeren) Einträge für JEDE Spalte
    eintraege_pro_spalte = df.notna().sum()
    
    # Zählt ALLE echten Einträge im gesamten Datensatz zusammen
    total_eintraege_datensatz = eintraege_pro_spalte.sum()
    
    # Filtert die Spalten heraus, die unsere Grenze NICHT schaffen
    spalten_die_wegfallen = eintraege_pro_spalte[eintraege_pro_spalte < mindest_zeilen]
    
    # Mathematik für den Bericht
    anzahl_spalten_weg = len(spalten_die_wegfallen)
    verlorene_eintraege_absolut = spalten_die_wegfallen.sum()
    
    # Verhindert Fehler, falls der Datensatz komplett leer sein sollte
    if total_eintraege_datensatz > 0:
        verlorene_eintraege_relativ = (verlorene_eintraege_absolut / total_eintraege_datensatz) * 100
    else:
        verlorene_eintraege_relativ = 0.0
        
    # 2. Den Bericht sauber ausdrucken
    print(f"🔍 SIMULATION: {prozent_grenze * 100:.1f}% Füllgrenze")
    print(f"(Mindestens benötigte definierte Werte pro Spalte: {mindest_zeilen:,})".replace(',', '.'))
    print("=" * 60)
    print(f"❌ Spalten, die wegfallen:        {anzahl_spalten_weg} (von insgesamt {len(df.columns)})")
    print(f"📉 Verlorene Einträge (absolut):  {verlorene_eintraege_absolut:,}".replace(',', '.'))
    print(f"📊 Verlorene Einträge (relativ):  {verlorene_eintraege_relativ:.2f}% aller Daten im Set")
    print("-" * 60)
    
    # Profi-Trick: Wenn es extrem viele Spalten sind, zeigen wir nur die ersten 50, 
    # damit dein Bildschirm nicht überflutet wird.
    spalten_liste = spalten_die_wegfallen.index.tolist()
    print(f"Namen der betroffenen Spalten:")
    print(spalten_liste)
        
    print("\n") # Leerzeile für den Abstand zum nächsten Test


# ==========================================
# 3. HIER KANNST DU JETZT RUMTESTEN!
# ==========================================

# Teste 5% (Der "Müllschlucker")
teste_fuellgrenze(df, 0.05)

# Teste 20% (Der "Sweet Spot")
teste_fuellgrenze(df, 0.20)

# Teste 50% (Der "Strenge Standard")
teste_fuellgrenze(df, 0.50)

🔍 SIMULATION: 5.0% Füllgrenze
(Mindestens benötigte definierte Werte pro Spalte: 225.061)
❌ Spalten, die wegfallen:        86 (von insgesamt 180)
📉 Verlorene Einträge (absolut):  1.675.873
📊 Verlorene Einträge (relativ):  1.06% aller Daten im Set
------------------------------------------------------------
Namen der betroffenen Spalten:
['packaging_text', 'origins_en', 'manufacturing_places_tags', 'emb_codes', 'first_packaging_code_geo', 'cities_tags', 'purchase_places', 'no_nutrition_data', 'additives', 'owner', 'data_quality_errors_tags', 'energy-from-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-

In [6]:
teste_fuellgrenze(df, 0.1)

🔍 SIMULATION: 10.0% Füllgrenze
(Mindestens benötigte definierte Werte pro Spalte: 450.122)
❌ Spalten, die wegfallen:        124 (von insgesamt 180)
📉 Verlorene Einträge (absolut):  11.826.429
📊 Verlorene Einträge (relativ):  7.47% aller Daten im Set
------------------------------------------------------------
Namen der betroffenen Spalten:
['packaging_en', 'packaging_text', 'origins_en', 'manufacturing_places_tags', 'emb_codes', 'first_packaging_code_geo', 'cities_tags', 'purchase_places', 'allergens', 'traces', 'no_nutrition_data', 'additives', 'brand_owner', 'owner', 'data_quality_errors_tags', 'energy-from-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g'

In [7]:
teste_fuellgrenze(df, 0.09)

🔍 SIMULATION: 9.0% Füllgrenze
(Mindestens benötigte definierte Werte pro Spalte: 405.110)
❌ Spalten, die wegfallen:        124 (von insgesamt 180)
📉 Verlorene Einträge (absolut):  11.826.429
📊 Verlorene Einträge (relativ):  7.47% aller Daten im Set
------------------------------------------------------------
Namen der betroffenen Spalten:
['packaging_en', 'packaging_text', 'origins_en', 'manufacturing_places_tags', 'emb_codes', 'first_packaging_code_geo', 'cities_tags', 'purchase_places', 'allergens', 'traces', 'no_nutrition_data', 'additives', 'brand_owner', 'owner', 'data_quality_errors_tags', 'energy-from-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g',

In [8]:
teste_fuellgrenze(df, 0.08)

🔍 SIMULATION: 8.0% Füllgrenze
(Mindestens benötigte definierte Werte pro Spalte: 360.097)
❌ Spalten, die wegfallen:        122 (von insgesamt 180)
📉 Verlorene Einträge (absolut):  11.081.891
📊 Verlorene Einträge (relativ):  7.00% aller Daten im Set
------------------------------------------------------------
Namen der betroffenen Spalten:
['packaging_text', 'origins_en', 'manufacturing_places_tags', 'emb_codes', 'first_packaging_code_geo', 'cities_tags', 'purchase_places', 'traces', 'no_nutrition_data', 'additives', 'brand_owner', 'owner', 'data_quality_errors_tags', 'energy-from-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 

In [9]:
teste_fuellgrenze(df, 0.07)

🔍 SIMULATION: 7.0% Füllgrenze
(Mindestens benötigte definierte Werte pro Spalte: 315.085)
❌ Spalten, die wegfallen:        117 (von insgesamt 180)
📉 Verlorene Einträge (absolut):  9.395.791
📊 Verlorene Einträge (relativ):  5.93% aller Daten im Set
------------------------------------------------------------
Namen der betroffenen Spalten:
['packaging_text', 'origins_en', 'manufacturing_places_tags', 'emb_codes', 'first_packaging_code_geo', 'cities_tags', 'purchase_places', 'traces', 'no_nutrition_data', 'additives', 'owner', 'data_quality_errors_tags', 'energy-from-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100

In [10]:
teste_fuellgrenze(df, 0.06)

🔍 SIMULATION: 6.0% Füllgrenze
(Mindestens benötigte definierte Werte pro Spalte: 270.073)
❌ Spalten, die wegfallen:        110 (von insgesamt 180)
📉 Verlorene Einträge (absolut):  7.467.612
📊 Verlorene Einträge (relativ):  4.71% aller Daten im Set
------------------------------------------------------------
Namen der betroffenen Spalten:
['packaging_text', 'origins_en', 'manufacturing_places_tags', 'emb_codes', 'first_packaging_code_geo', 'cities_tags', 'purchase_places', 'traces', 'no_nutrition_data', 'additives', 'owner', 'data_quality_errors_tags', 'energy-from-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100

In [11]:
teste_fuellgrenze(df, 0.05)

🔍 SIMULATION: 5.0% Füllgrenze
(Mindestens benötigte definierte Werte pro Spalte: 225.061)
❌ Spalten, die wegfallen:        86 (von insgesamt 180)
📉 Verlorene Einträge (absolut):  1.675.873
📊 Verlorene Einträge (relativ):  1.06% aller Daten im Set
------------------------------------------------------------
Namen der betroffenen Spalten:
['packaging_text', 'origins_en', 'manufacturing_places_tags', 'emb_codes', 'first_packaging_code_geo', 'cities_tags', 'purchase_places', 'no_nutrition_data', 'additives', 'owner', 'data_quality_errors_tags', 'energy-from-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-

In [12]:
a = ['packaging_text', 'origins_en', 'emb_codes', 'first_packaging_code_geo', 'cities_tags', 'purchase_places', 'traces', 'no_nutrition_data', 'additives', 'owner', 'data_quality_errors_tags', 'energy-from-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-6-fat_100g', 'alpha-linolenic-acid_100g', 'eicosapentaenoic-acid_100g', 'docosahexaenoic-acid_100g', 'linoleic-acid_100g', 'arachidonic-acid_100g', 'gamma-linolenic-acid_100g', 'dihomo-gamma-linolenic-acid_100g', 'oleic-acid_100g', 'elaidic-acid_100g', 'gondoic-acid_100g', 'mead-acid_100g', 'erucic-acid_100g', 'nervonic-acid_100g', 'trans-fat_100g', 'sucrose_100g', 'glucose_100g', 'fructose_100g', 'galactose_100g', 'lactose_100g', 'maltose_100g', 'maltodextrins_100g', 'psicose_100g', 'starch_100g', 'polyols_100g', 'erythritol_100g', 'isomalt_100g', 'maltitol_100g', 'sorbitol_100g', 'soluble-fiber_100g', 'polydextrose_100g', 'insoluble-fiber_100g', 'casein_100g', 'serum-proteins_100g', 'nucleotides_100g', 'added-salt_100g', 'beta-carotene_100g', 'vitamin-d_100g', 'vitamin-k_100g', 'vitamin-b1_100g', 'vitamin-b2_100g', 'vitamin-pp_100g', 'vitamin-b6_100g', 'vitamin-b9_100g', 'folates_100g', 'vitamin-b12_100g', 'biotin_100g', 'pantothenic-acid_100g', 'silica_100g', 'bicarbonate_100g', 'chloride_100g', 'copper_100g', 'manganese_100g', 'fluoride_100g', 'selenium_100g', 'chromium_100g', 'molybdenum_100g', 'iodine_100g', 'caffeine_100g', 'taurine_100g', 'methylsulfonylmethane_100g', 'ph_100g', 'collagen-meat-protein-ratio_100g', 'cocoa_100g', 'chlorophyl_100g', 'carbon-footprint_100g', 'glycemic-index_100g', 'water-hardness_100g', 'choline_100g', 'phylloquinone_100g', 'beta-glucan_100g', 'inositol_100g', 'carnitine_100g', 'sulphate_100g', 'nitrate_100g', 'acidity_100g', 'carbohydrates-total_100g', 'water_100g', 'manufacturing_places_combined']
b = ['packaging_text', 'origins_en', 'emb_codes', 'first_packaging_code_geo', 'cities_tags', 'purchase_places', 'no_nutrition_data', 'additives', 'owner', 'data_quality_errors_tags', 'energy-from-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-6-fat_100g', 'alpha-linolenic-acid_100g', 'eicosapentaenoic-acid_100g', 'docosahexaenoic-acid_100g', 'linoleic-acid_100g', 'arachidonic-acid_100g', 'gamma-linolenic-acid_100g', 'dihomo-gamma-linolenic-acid_100g', 'oleic-acid_100g', 'elaidic-acid_100g', 'gondoic-acid_100g', 'mead-acid_100g', 'erucic-acid_100g', 'nervonic-acid_100g', 'trans-fat_100g', 'maltodextrins_100g', 'psicose_100g', 'erythritol_100g', 'isomalt_100g', 'maltitol_100g', 'sorbitol_100g', 'soluble-fiber_100g', 'polydextrose_100g', 'insoluble-fiber_100g', 'casein_100g', 'serum-proteins_100g', 'nucleotides_100g', 'added-salt_100g', 'vitamin-k_100g', 'folates_100g', 'biotin_100g', 'silica_100g', 'bicarbonate_100g', 'chloride_100g', 'fluoride_100g', 'chromium_100g', 'molybdenum_100g', 'caffeine_100g', 'taurine_100g', 'methylsulfonylmethane_100g', 'ph_100g', 'collagen-meat-protein-ratio_100g', 'cocoa_100g', 'chlorophyl_100g', 'carbon-footprint_100g', 'glycemic-index_100g', 'water-hardness_100g', 'choline_100g', 'beta-glucan_100g', 'inositol_100g', 'carnitine_100g', 'sulphate_100g', 'nitrate_100g', 'acidity_100g', 'carbohydrates-total_100g', 'manufacturing_places_combined']
c = list(set(a) -set(b))

print(c)

['vitamin-b2_100g', 'water_100g', 'lactose_100g', 'glucose_100g', 'vitamin-b1_100g', 'iodine_100g', 'vitamin-d_100g', 'vitamin-b6_100g', 'vitamin-pp_100g', 'manganese_100g', 'vitamin-b12_100g', 'copper_100g', 'traces', 'sucrose_100g', 'beta-carotene_100g', 'selenium_100g', 'vitamin-b9_100g', 'pantothenic-acid_100g', 'starch_100g', 'galactose_100g', 'polyols_100g', 'fructose_100g', 'maltose_100g', 'phylloquinone_100g']


In [13]:
import pandas as pd

# 1. Wir definieren deine gewählte Grenze (6 %)
fuell_quote = 0.06

# 2. Wir berechnen die absolute Anzahl an benötigten Zeilen
mindest_eintraege = int(len(df) * fuell_quote)

print(f"Bereinigung gestartet...")
print(f"Jede Spalte muss mindestens {mindest_eintraege:,} definierte Werte haben.".replace(',', '.'))
print("--------------------------------------------------")

# Festhalten der alten Spaltenanzahl für den Vergleich
spalten_vorher = len(df.columns)

# 3. DAS LÖSCHEN:
# Wir überschreiben unser 'df' mit der bereinigten Version.
# axis=1 zielt auf die Spalten ab, thresh setzt die Grenze.
df = df.dropna(axis=1, thresh=mindest_eintraege)

# 4. Zusammenfassung ausdrucken
spalten_nachher = len(df.columns)
geloescht = spalten_vorher - spalten_nachher

print(f"✅ Bereinigung erfolgreich!")
print(f"Es wurden {geloescht} fast leere Spalten gelöscht.")
print(f"Neue Spaltenanzahl: {spalten_nachher} (vorher: {spalten_vorher})")

Bereinigung gestartet...
Jede Spalte muss mindestens 270.073 definierte Werte haben.
--------------------------------------------------
✅ Bereinigung erfolgreich!
Es wurden 110 fast leere Spalten gelöscht.
Neue Spaltenanzahl: 70 (vorher: 180)


In [14]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 4501223 entries, 0 to 4501354
Data columns (total 70 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   code                            4501223 non-null  object 
 1   url                             4501223 non-null  object 
 2   creator                         4501217 non-null  object 
 3   created_t                       4501223 non-null  int64  
 4   created_datetime                4501223 non-null  object 
 5   last_modified_t                 4501223 non-null  int64  
 6   last_modified_datetime          4501223 non-null  object 
 7   last_modified_by                4362662 non-null  object 
 8   last_updated_t                  4501071 non-null  float64
 9   last_updated_datetime           4501071 non-null  object 
 10  product_name                    4169000 non-null  object 
 11  packaging_en                    378941 non-null   object 
 12  brand

In [15]:
df.to_csv(
    'openfoodfacts_ueber6prozent.csv.zip', 
    index=False, 
    sep='\t', 
    encoding='utf-8',
    compression="zip",
)